# Calibration of PAPYRUS: a pyramid WFS, and its deformable mirror.

Papyrus was a Pyramid-based AO system installed in the T152 telescope in OHP, France. It had an Alpao241 DM

The calibration procedure is as follows:
1. We load the data, and pre-process it to facilitate the calibration.
2. We make a first rough calibration to place the pupils in the correct positions, and calibrate PAPYRUS's rooftop prism against the bench reference frame's quadrant flux.
3. We calibrate the static amplitude and phase from the bench
4. We define the deformable mirror and perform a rough alignment to determine rotations, flips and signs
5. Using the fully differentiable WFS and DM models we fit all the degrees of freedom to compute the misregistration

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from AI4AO import PyramidWFS, DeformableMirror, TwinCalibrator

from mmengine import Config
import numpy as np
import matplotlib.pyplot as plt

You can download the data to calibrate from here

[Link to files](https://nuage.osupytheas.fr/s/TqpdFYx2rKsqgLX)

We load the bench interaction matrix, the modes-to-commands matrix `M2C`, and a dedicated bench reference frame acquired for pupil positioning.

In [ ]:
papyrus_iMat = np.load("../../Data/Papyrus/Papyrus_iMat.npy")
papyrus_iMat = torch.from_numpy(papyrus_iMat.astype(np.float32)).to(device = device, dtype=torch.float32)

M2C = np.load("../../Data/Papyrus/M2C.npy")
M2C = torch.from_numpy(M2C).to(device = device, dtype = torch.float32)

reference_frame = np.load('../../Data/Papyrus/Reference_frame.npy')
reference_frame = torch.from_numpy(reference_frame).to(device = device, dtype = torch.float32)
reference_frame = reference_frame.view(240,240)
reference_frame /= reference_frame.sum()

In [ ]:
paramfile = 'Papyrus_params.py'  # file of experimental parameters

# Config extraction

WFSParams = Config.fromfile(paramfile)['WFSParams']
WFSParams["centralObstruction"] = 0.
WFSParams["useNoise"] = False

DMParams = Config.fromfile(paramfile)['DMParams']


In [ ]:
wfs = PyramidWFS(WFSParams, device)

calibrator = TwinCalibrator(wfs, dm=None, device=device)

## Pupil positioning

We start the calibration by placing the pupils in the correct positions, using `TwinCalibrator.fit_pupil_to_reference`. It:
1. Updates the position of the pupils with `wfs.BuildMask()`
2. Computes a reference intensity with `wfs.BuildReferenceIntensity()`
3. Compares it to the reference frame and backpropagates

It is possible that the pupils are not aligned on the first try and you can run the cell as many times as you need. Also, you can change `n_iter` to have more or less iterations, and `lr` to change the learning rate of the optimizer.

In [ ]:
final_loss = calibrator.fit_pupil_to_reference(
    reference_frame,
    wfs.parameters(),
    lr=1e-3,
    n_iter=200,
)

PAPYRUS's pyramid mask includes a rooftop prism. We calibrate its `wfs.rooftop` parameter separately with `TwinCalibrator.fit_rooftop`, which matches the flux summed within each of the four quadrants of the reference frame instead of a pixel-wise fit.

In [ ]:
final_loss = calibrator.fit_rooftop(reference_frame, lr=1e-1, n_iter=100)

## DM parameters

`moffatParam`: Corresponds to the moffat exponent which interpolates between a Cauchy and Gaussian shapes for the influence functions.

`signedAmplitude`: Amplitude in OPD of the DM. Can be positive or negative, depending on convention.

`Flips`: Flip left-right or top-bottom. If you don't know them, don't worry as there is a rough calibration step that can find the best configuration.

`misreg`: Dictionary containing the misregistrations of the DM. These are compatible with OOPAO's misreg.

If you don't know the correct flips, rotations, and sign of the DM, `calibrator.rough_calibrate_dm(bench_iMat, M2C)` performs an automatic selection of the best starting values.

In [ ]:
param = {}
param['rotationAngle'] = 0
# shift X in m
param['shiftX'] = 0
# shift Y in m
param['shiftY'] = 0
# amamorphosis angle in degrees
param['anamorphosisAngle'] = 0
# normal scaling in % of diameter
param['tangentialScaling'] = 0
# radial scaling in % of diameter
param['radialScaling'] = 0

dm = DeformableMirror(WFSParams,
                    DMParams,
                    device=device,
                    misreg=param)

calibrator.dm = dm
calibrator.rough_calibrate_dm(papyrus_iMat, M2C)


We define the static amplitude and phase maps for the bench. These will get optimized along with the misreg of the DM. To avoid overfitting or falling into local minima, the training is set such that these maps are not updated at the beginning and slowly start getting more and more importance as the optimization progresses.

In [ ]:
ref_pupil, ref_phase = calibrator.init_static_offsets()

Always check on some modes to see if the rough alignment makes sense! You can change `idx` to see through some modes.

In [ ]:
index = list(range(0,190,5))

calibrator.sanity_check_plot(papyrus_iMat, M2C, index, idx=3)

## Optimization of misregistration, WFS parameters and static amplitude and phase

We define the optimization problem to be: find the best parameters for the DM, WFS and offsets to produce a synthetic interaction matrix as close as possible to the one measured on the bench, using `calibrator.fit_dm_and_offsets`.

Given that the main parameters to be trained are the DM's, the learning rate is set higher than for the WFS, which was previously optimized. For the static amplitude and phase, to avoid overfitting, the optimization starts with an extremely low learning rate for them, so the optimizer prioritizes changing the parameters of the DM and WFS first. The learning rate for the offsets is then gradually increased to match that of the DM and WFS.

In [ ]:
final_loss, original_positons, transformed_positons = calibrator.fit_dm_and_offsets(
    papyrus_iMat,
    M2C,
    index,
    n_iter=200,
    lr_dm=1e-2,
    lr_wfs=1e-3,
    lr_offset_start=-10,
    lr_offset_end=-2,
    fit_static_offsets=True,
    plot_mode_idx=4,
)

You can check the retrieved actuator positions, the static amplitude and phase values.

In [ ]:
calibrator.plot_actuator_and_offsets(original_positons, transformed_positons)

We now compute a new interaction matrix with the fitted parameters.

In [ ]:
calibrator.rebuild_reconstruction_matrix(M2C)

You can change the index to see how well the algorithm worked to fit the parameters.

In [ ]:
calibrator.plot_fit_residual(papyrus_iMat, idx=0)

In [ ]:
cov = calibrator.crosstalk_diagnostic(papyrus_iMat)

In [ ]:
PATH_WFS, PATH_DM = calibrator.save("Papyrus", data_dir = "../../Data")

In [ ]:
wfs = PyramidWFS(WFSParams, device)
dm = DeformableMirror(WFSParams,
                    DMParams,
                    device=device)

calibrator.wfs = wfs
calibrator.dm = dm
calibrator.load("Papyrus", data_dir = "../../Data")

calibrator.rebuild_reconstruction_matrix(M2C)
cov = calibrator.crosstalk_diagnostic(papyrus_iMat)